# GAT-GRU + PPO trong Digital Twin — v1

Notebook này không dùng lookup và không dùng MLP sinh toàn bộ next-state. Digital Twin tách riêng: workload ngoại sinh, lifecycle replica, queue/backlog, CPU theo work conservation, phân phối latency theo từng service và lớp quan sát Jaeger 10%. Test calibration bị khóa cho đến bước đánh giá cuối.


In [ ]:
%pip install -q "gymnasium==1.2.0" "stable-baselines3>=2.4,<3" "scikit-learn>=1.3"


## 1. Cấu hình, contract và tái lập


In [ ]:
import os, sys, json, random, hashlib, mathfrom dataclasses import dataclass, fieldfrom pathlib import Pathfrom typing import Any, Dict, List, Tuple, Optionalfrom collections import defaultdict, dequeimport numpy as npimport pandas as pdimport torchfrom torch import nnimport gymnasium as gymfrom gymnasium import spacesfrom sklearn.neighbors import LocalOutlierFactorfrom sklearn.preprocessing import MinMaxScalerimport joblibfrom stable_baselines3 import PPOfrom stable_baselines3.common.vec_env import DummyVecEnv, VecNormalizefrom stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallbackwarnings.filterwarnings('ignore')SEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed_all(SEED)DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {DEVICE}')print(f'Seed: {SEED}')

## 2. Nạp manifest hiệu chỉnh và khóa test


In [ ]:
PROTOCOL = 'digital_twin_calibration_v1'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def find_manifest():
    configured=os.environ.get('KAGGLE_TWIN_MANIFEST')
    if configured: return Path(configured).resolve()
    hits=list(Path('/kaggle/input').rglob('digital_twin_manifest.json')) if Path('/kaggle/input').exists() else []
    if len(hits)!=1: raise FileNotFoundError('Set KAGGLE_TWIN_MANIFEST to the unique exported manifest')
    return hits[0].resolve()

MANIFEST_PATH=find_manifest(); DATA_ROOT=MANIFEST_PATH.parent.resolve()
MANIFEST=json.loads(MANIFEST_PATH.read_text())
assert MANIFEST['protocol']==PROTOCOL
assert MANIFEST['plan']['sampling_percentage']==10
assert MANIFEST['plan']['replica_levels']==[1,2,3]
assert tuple(MANIFEST['plan']['controlled_services'])==CFG.controlled
assert MANIFEST['plan']['max_total_app_pods']==CFG.max_total_app_pods
assert MANIFEST['plan']['window_sec']==CFG.tick_sec*CFG.decision_ticks
if MANIFEST['missing_cells']: raise RuntimeError(f"Incomplete campaign: {len(MANIFEST['missing_cells'])} cells missing")

def inside(root, path):
    try: Path(path).resolve().relative_to(Path(root).resolve()); return True
    except ValueError: return False

def runs_for(split, allow_test=False):
    if split=='test' and not allow_test: raise PermissionError('Calibration test is sealed')
    rows=[]
    for entry in MANIFEST['runs']:
        if entry['split']!=split: continue
        folder=(DATA_ROOT/entry['directory']).resolve()
        if not inside(DATA_ROOT,folder): raise ValueError('Manifest path escape')
        for name,digest in entry['raw_sha256'].items():
            path=folder/name
            if not path.is_file() or sha256(path)!=digest: raise ValueError(f'Changed calibration file: {path}')
        rows.append((entry,folder))
    if not rows: raise ValueError(f'No {split} runs')
    return rows
TRAIN_RUNS=runs_for('train'); VAL_RUNS=runs_for('val')
print('train runs=',len(TRAIN_RUNS),'validation runs=',len(VAL_RUNS),'test remains sealed')


## 3. Hiệu chỉnh twin
CPU được fit bằng mô hình không âm `idle × ready + cost/request × throughput`. Capacity lấy từ miền throughput/replica quan sát. Latency được mô hình bằng hỗn hợp LogNormal có điều kiện theo service, request rate, ready replica và queue pressure; mô hình không dự đoán replica hay workload.


In [ ]:
# V2 collection data must not enter the legacy 10s fitting pipeline.
for entry, folder in TRAIN_RUNS + VAL_RUNS:
    metadata = json.loads((folder / 'meta.json').read_text())
    if metadata.get('calibration_schema_version', 1) >= 2:
        raise RuntimeError(
            'Calibration v2: review calibration_diagnostics.json and shared calibration.py. '
            'Own-service-time/capture evidence and response-dependent Twin adapter required; '
            'legacy 10s fitting is disabled for this dataset.')

SERVICES=list(fd.SERVICES); SVC={s:i for i,s in enumerate(SERVICES)}; EDGES=list(fd.STATIC_EDGES)
def load_tables(runs):
    ticks=[]; spans=[]; edges=[]; startup=[]
    for entry,folder in runs:
        node=pd.read_csv(folder/'node_metrics.csv')
        svc=pd.read_csv(folder/'service_metrics.csv')
        tick=node.merge(svc,on=['timestamp','service'],how='left')
        tick['request_count']=tick['request_count'].fillna(0)
        tick['error_count']=tick['error_count'].fillna(0)
        tick['rps']=tick['request_count']/(CFG.tick_sec*CFG.sampling)
        tick['run']=entry['directory']; tick['scenario']=entry['scenario']; tick['split']=entry['split']
        ticks.append(tick)
        raw=pd.read_csv(folder/'jaeger_server_spans.csv')
        raw=raw.merge(tick[['timestamp','service','rps','replicas','cpu_cores']],
                      left_on=['bucket_timestamp','service'],right_on=['timestamp','service'],how='inner')
        cluster=tick.groupby('timestamp',as_index=False).replicas.sum().rename(columns={'timestamp':'bucket_timestamp','replicas':'cluster_replicas'})
        raw=raw.merge(cluster,on='bucket_timestamp',how='left')
        raw['run']=entry['directory']; spans.append(raw)
        edge=pd.read_csv(folder/'edge_metrics.csv'); edge['run']=entry['directory']; edges.append(edge)
        for row in entry['capacity_delay_audit']:
            if row['desired']>1 and row['first_observed_available_delay_sec'] is not None:
                startup.append(dict(service=row['service'],desired=row['desired'],delay=row['first_observed_available_delay_sec']))
    return pd.concat(ticks,ignore_index=True),pd.concat(spans,ignore_index=True),pd.concat(edges,ignore_index=True),pd.DataFrame(startup)

train_ticks,train_spans,train_edges,startup_train=load_tables(TRAIN_RUNS)
val_ticks,val_spans,val_edges,startup_val=load_tables(VAL_RUNS)
for frame in (train_ticks,val_ticks): frame['per_replica']=frame['rps']/frame['replicas'].clip(lower=1)

def fit_cpu_binomial_eiv(rows,sampling,tick_sec):
    ready=rows.replicas.to_numpy(float); sampled=rows.rps.to_numpy(float); cpu=rows.cpu_cores.to_numpy(float)
    noise_var=np.maximum(sampled,0)*(1-sampling)/(sampling*tick_sec)
    a=float(ready@ready); b=float(ready@sampled); c=float(sampled@sampled-noise_var.sum())
    d=float(ready@cpu); e=float(sampled@cpu); det=a*c-b*b
    if c<=0 or det<=1e-10*max(1.,a*c): raise RuntimeError('CPU EIV corrected Gram is singular or non-positive')
    candidates=[np.zeros(2,dtype=float)]
    if a>1e-12: candidates.append(np.array([max(0,d/a),0.]))
    if c>1e-12: candidates.append(np.array([0.,max(0,e/c)]))
    beta=np.array([(d*c-e*b)/det,(e*a-d*b)/det])
    if np.all(beta>=0): candidates.append(beta)
    def corrected_score(beta):
        residual=cpu-beta[0]*ready-beta[1]*sampled
        return float(residual@residual-beta[1]**2*noise_var.sum())
    return min(candidates,key=corrected_score)

PARAMS={'sampling':CFG.sampling,'tick_sec':CFG.tick_sec,'services':SERVICES,
        'capacity_estimator':'legacy_disabled','cpu_estimator':'binomial_eiv_corrected_score_v1'}
PARAMS['cluster_replicas_q99']=float(train_ticks.groupby(['run','timestamp']).replicas.sum().quantile(.99))
PARAMS['mu_rps_per_replica']={}; PARAMS['capacity_identified']={}
PARAMS['cpu']={}; PARAMS['error']={}; PARAMS['resources']={}; PARAMS['support']={}; PARAMS['startup_delay']={}
for service in SERVICES:
    rows=train_ticks[train_ticks.service==service].copy()
    if service=='redis-cart':
        PARAMS['mu_rps_per_replica'][service]=1e6; PARAMS['capacity_identified'][service]=True
        PARAMS['cpu'][service]={'idle_per_replica':float(rows.cpu_cores.median()),'core_per_rps':0.0}
        PARAMS['error'][service]={'base':0.0,'overflow_slope':0.0}
        PARAMS['resources'][service]={'mem_per_replica':float((rows.mem_bytes/rows.replicas.clip(lower=1)).median()),'net_rx_per_rps':0.0,'net_tx_per_rps':0.0}
        PARAMS['support'][service]={'rps_q01':0.0,'rps_q99':0.0,'rho_q99':0.0}; PARAMS['startup_delay'][service]=[]
        continue
    raise RuntimeError(f'Legacy capacity fitting disabled for {service}: latency alone cannot identify mu; use calibration.py with validated evidence.')
    capacity=PARAMS['mu_rps_per_replica'][service]*rows.replicas.clip(lower=1)
    overflow=((rows.rps-capacity).clip(lower=0)/capacity.clip(lower=1e-3)).to_numpy()
    observed_error=(rows.error_count/rows.request_count.clip(lower=1)).fillna(0).to_numpy()
    err_reg=LinearRegression(positive=True,fit_intercept=False).fit(np.c_[np.ones(len(rows)),overflow],observed_error)
    PARAMS['error'][service]={'base':float(err_reg.coef_[0]),'overflow_slope':float(err_reg.coef_[1])}
    PARAMS['resources'][service]={
      'mem_per_replica':float((rows.mem_bytes/rows.replicas.clip(lower=1)).median()),
      'net_rx_per_rps':float((rows.net_rx_bps/rows.rps.clip(lower=.1)).median()),
      'net_tx_per_rps':float((rows.net_tx_bps/rows.rps.clip(lower=.1)).median())}
    PARAMS['support'][service]={'rps_q01':float(rows.rps.quantile(.01)),'rps_q99':float(rows.rps.quantile(.99)),
                                'rho_q99':float((rows.rps/capacity.clip(lower=1e-3)).quantile(.99))}
    delays=startup_train[startup_train.service==service].delay.to_numpy(float)
    PARAMS['startup_delay'][service]=delays.tolist()

PARAMS['fanout']={}
source_rates=train_ticks[['run','timestamp','service','rps']].rename(columns={'service':'source','rps':'source_rps'})
edge=train_edges.merge(source_rates,on=['run','timestamp','source'],how='left')
edge['call_rate']=edge.call_count/(CFG.tick_sec*CFG.sampling)
for source,target in EDGES:
    rows=edge[(edge.source==source)&(edge.target==target)&(edge.source_rps>0)]
    PARAMS['fanout'][f'{source}->{target}']=float((rows.call_rate/rows.source_rps).median()) if len(rows) else 0.0

def add_mdn_features(spans):
    out=spans.copy(); mu=out.service.map(PARAMS['mu_rps_per_replica']).astype(float)
    capacity=mu*out.replicas.clip(lower=1)
    out['rho']=(out.rps/capacity).clip(0,4)
    out['overflow']=((out.rps-capacity).clip(lower=0)/capacity.clip(lower=1e-3)).clip(0,4)
    out['service_id']=out.service.map(SVC)
    out['log_latency']=np.log(out.latency_ms.clip(lower=.01))
    return out[np.isfinite(out[['rps','replicas','rho','overflow','cluster_replicas','log_latency']]).all(axis=1)]
train_mdn=add_mdn_features(train_spans); val_mdn=add_mdn_features(val_spans)
print('ticks=',len(train_ticks),'train spans=',len(train_mdn),'val spans=',len(val_mdn))


In [ ]:
class LatencyMDN(nn.Module):
    def __init__(self,n_services,n_components):
        super().__init__(); self.k=n_components; self.embed=nn.Embedding(n_services,8)
        self.body=nn.Sequential(nn.Linear(13,64),nn.SiLU(),nn.Linear(64,64),nn.SiLU())
        self.head=nn.Linear(64,3*n_components)
    def forward(self,service_id,x):
        raw=self.head(self.body(torch.cat([self.embed(service_id),x],dim=1)))
        logits,loc,scale=raw.chunk(3,dim=1)
        return logits,loc,torch.nn.functional.softplus(scale)+0.03

def tensors(frame):
    x=np.c_[np.log1p(frame.rps),frame.replicas/CFG.max_replicas,frame.rho,frame.overflow,
            frame.cluster_replicas/CFG.max_total_app_pods].astype('float32')
    return (torch.as_tensor(frame.service_id.to_numpy(),dtype=torch.long),
            torch.as_tensor(x),torch.as_tensor(frame.log_latency.to_numpy('float32')))
def mdn_nll(logits,loc,scale,y):
    log_prob=torch.distributions.Normal(loc,scale).log_prob(y[:,None])+torch.log_softmax(logits,dim=1)
    return -torch.logsumexp(log_prob,dim=1).mean()

model=LatencyMDN(len(SERVICES),CFG.mdn_components).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
sid,x,y=tensors(train_mdn); dataset=torch.utils.data.TensorDataset(sid,x,y)
loader=torch.utils.data.DataLoader(dataset,batch_size=CFG.mdn_batch,shuffle=True,generator=torch.Generator().manual_seed(SEED))
best=None; best_val=float('inf'); history=[]
vsid,vx,vy=(z.to(DEVICE) for z in tensors(val_mdn))
for epoch in range(CFG.mdn_epochs):
    model.train(); losses=[]
    for bsid,bx,by in loader:
        bsid,bx,by=bsid.to(DEVICE),bx.to(DEVICE),by.to(DEVICE)
        loss=mdn_nll(*model(bsid,bx),by); optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5); optimizer.step(); losses.append(loss.item())
    model.eval()
    with torch.no_grad(): val=float(mdn_nll(*model(vsid,vx),vy).cpu())
    history.append({'epoch':epoch,'train_nll':float(np.mean(losses)),'val_nll':val})
    if val<best_val: best_val=val; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best); model.eval(); pd.DataFrame(history).to_csv(OUT/'mdn_history.csv',index=False)
torch.save({'state_dict':best,'params':PARAMS,'config':CFG.__dict__},OUT/'digital_twin.pt')
(OUT/'digital_twin_params.json').write_text(json.dumps(PARAMS,indent=2))
print('best validation NLL=',best_val)


## 4. Validation bắt buộc trước PPO


In [ ]:
@torch.no_grad()
def sample_mdn(service_ids,x,n=64,rng=None):
    rng=np.random.default_rng(SEED) if rng is None else rng
    sid=torch.as_tensor(service_ids,dtype=torch.long,device=DEVICE)
    xx=torch.as_tensor(x,dtype=torch.float32,device=DEVICE)
    logits,loc,scale=model(sid,xx); prob=torch.softmax(logits,1).cpu().numpy()
    loc=loc.cpu().numpy(); scale=scale.cpu().numpy(); out=np.empty((len(x),n))
    for i in range(len(x)):
        component=rng.choice(model.k,size=n,p=prob[i])
        out[i]=np.exp(rng.normal(loc[i,component],scale[i,component]))
    return out

vsid_cpu,vx_cpu,vy_cpu=tensors(val_mdn); draws=sample_mdn(vsid_cpu.numpy(),vx_cpu.numpy(),128)
truth=np.exp(vy_cpu.numpy()); median=np.median(draws,axis=1); lo=np.quantile(draws,.05,axis=1); hi=np.quantile(draws,.95,axis=1)
train_medians=train_mdn.assign(latency=np.exp(train_mdn.log_latency)).groupby('service_id').latency.median()
baseline=val_mdn.service_id.map(train_medians).to_numpy(float)
diag=pd.DataFrame({'service':val_mdn.service.to_numpy(),'truth':truth,'median':median,'lo':lo,'hi':hi,'baseline':baseline})
diag['covered']=(diag.truth>=diag.lo)&(diag.truth<=diag.hi); diag['abs_error']=(diag.truth-diag['median']).abs(); diag['baseline_abs_error']=(diag.truth-diag.baseline).abs()
service_metrics=diag.groupby('service').agg(samples=('truth','size'),coverage90=('covered','mean'),mae_ms=('abs_error','mean'),baseline_mae_ms=('baseline_abs_error','mean')).reset_index()
service_metrics['improvement']=1-service_metrics.mae_ms/service_metrics.baseline_mae_ms.clip(lower=1e-6)
service_metrics.to_csv(OUT/'latency_validation_by_service.csv',index=False)
coverage=float(diag.covered.mean()); mae=float(diag.abs_error.mean()); base_mae=float(diag.baseline_abs_error.mean())
frontend=diag[diag.service=='frontend']; slo_rate_error=abs(float((draws[val_mdn.service.to_numpy()=='frontend']>CFG.slo_ms).mean())-float((frontend.truth>CFG.slo_ms).mean()))
level_support=lambda frame: all(set(frame[frame.service==s].replicas.astype(int))>=set(range(1,CFG.max_replicas+1)) for s in CFG.controlled)
startup_services=set(startup_train.service)&set(CFG.controlled)
adj=np.zeros((len(SERVICES),len(SERVICES)),int); fan=np.zeros_like(adj,dtype=float)
for a,b in EDGES: adj[SVC[a],SVC[b]]=1; fan[SVC[a],SVC[b]]=PARAMS['fanout'][f'{a}->{b}']
degree=adj.sum(0); queue=deque(np.flatnonzero(degree==0).tolist()); topo=[]
while queue:
    i=queue.popleft(); topo.append(i)
    for j in np.flatnonzero(adj[i]):
        degree[j]-=1
        if degree[j]==0: queue.append(int(j))
if len(topo)!=len(SERVICES): raise RuntimeError('Static service graph contains a cycle')
def graph_rates(frontend_rps):
    out=np.zeros(len(SERVICES)); out[SVC['frontend']]=frontend_rps
    for i in topo:
        for j in np.flatnonzero(adj[i]): out[j]+=out[i]*fan[i,j]
    return out
observed=val_ticks.pivot_table(index=['run','timestamp'],columns='service',values='rps').reindex(columns=SERVICES).dropna()
predicted=np.stack([graph_rates(value) for value in observed.frontend.to_numpy()])
graph_load_metrics=pd.DataFrame({'service':SERVICES,'normalized_mae':[float(np.mean(np.abs(predicted[:,i]-observed.iloc[:,i].to_numpy())/(observed.iloc[:,i].to_numpy()+1))) for i in range(len(SERVICES))]})
graph_load_metrics.to_csv(OUT/'graph_load_validation.csv',index=False)
GATES={
 'replica_support':level_support(train_ticks) and level_support(val_ticks),
 'startup_support':set(CFG.controlled)<=startup_services,
 'capacity_onset_observed':all(PARAMS['capacity_identified'][s] for s in CFG.controlled),
 'graph_fanout_support':all(PARAMS['fanout'][f'{a}->{b}']>0 for a,b in EDGES if b!='redis-cart'),
 'workload_support_identified':all(PARAMS['support'][s]['rps_q99']>PARAMS['support'][s]['rps_q01'] for s in CFG.controlled),
 'cluster_interaction_support':PARAMS['cluster_replicas_q99']>=len(SERVICES)+2,
 'graph_load_accuracy':all(row.normalized_mae<=.50 for row in graph_load_metrics.itertuples() if row.service in CFG.controlled),
 'finite_nll':np.isfinite(best_val),
 'interval_coverage':all(0.70<=row.coverage90<=0.99 for row in service_metrics.itertuples() if row.service in CFG.controlled),
 'latency_skill':all(row.improvement>0 for row in service_metrics.itertuples() if row.service in CFG.controlled),
 'frontend_slo_calibration':slo_rate_error<=0.10}
pd.Series(GATES,name='passed').to_csv(OUT/'digital_twin_gates.csv')
print({'coverage90':coverage,'mae_ms':mae,'baseline_mae_ms':base_mae,'frontend_slo_rate_error':slo_rate_error}); display(service_metrics); display(graph_load_metrics); print(pd.Series(GATES))
AUTO_GATES_PASS=all(GATES.values())
TWIN_APPROVED=False  # chỉ đổi True sau khi review plots, per-service metrics và held-out rollouts
print('AUTO_GATES_PASS=',AUTO_GATES_PASS,'TWIN_APPROVED=',TWIN_APPROVED)


## 5. Digital Twin Gymnasium: capacity → queue → performance → Jaeger observation


In [ ]:
class MicroserviceDigitalTwin(gym.Env):    """Grey-box Digital Twin for Proactive Microservice Autoscaling."""    metadata = {'render_modes': []}    def __init__(self, episodes, seed=42, sequential=False, predictor=None, lof_detector=None):        super().__init__()        self.episodes = episodes        self.sequential = sequential        self.rng = np.random.default_rng(seed)        self.episode_cursor = 0        self.twin = None        self.t = 0        self.state = None        self.reward_p95_ema = None        self.episode_step = 0        self.predictor = predictor        self.lof_detector = lof_detector        self.action_space = spaces.MultiDiscrete([3] * 10)        self.observation_space = spaces.Box(            low=0.0, high=1.0, shape=(228,), dtype=np.float32        )        self.all_services = [            'frontend', 'cartservice', 'checkoutservice', 'currencyservice',            'emailservice', 'paymentservice', 'productcatalogservice',            'recommendationservice', 'shippingservice', 'adservice', 'redis-cart'        ]        self.controlled = [            'frontend', 'cartservice', 'checkoutservice', 'currencyservice',            'emailservice', 'paymentservice', 'productcatalogservice',            'recommendationservice', 'shippingservice', 'adservice'        ]        self.n_all = 11        self.n_ctrl = 10        self.feature_min = np.array([0.0] * 13, dtype=np.float32)        self.feature_max = np.array([            8.0, 24576.0, 2.0, 1.0, 1.0, 500.0, 500.0, 1.0,            100.0, 20.0, 10.0, 10.0, 5.0        ], dtype=np.float32)        self.feature_range = self.feature_max - self.feature_min        self.feature_range[self.feature_range == 0] = 1.0        self.current_fields = [            'cpu_cores', 'mem_mib', 'cpu_util_request', 'cpu_throttle_ratio',            'cpu_sidecar_cores', 'latency_p50_ms', 'latency_p95_ms', 'error_rate',            'rps_in', 'rps_in_delta', 'rps_per_replica', 'replicas', 'restarts_delta'        ]        self.edge_feature_names = ['call_rate', 'call_ratio', 'edge_error_rate', 'edge_latency_p95_ms']        self.obs_edges = [            ('frontend', 'productcatalogservice'), ('frontend', 'currencyservice'),            ('frontend', 'cartservice'), ('frontend', 'recommendationservice'),            ('frontend', 'shippingservice'), ('frontend', 'checkoutservice'),            ('frontend', 'adservice'), ('recommendationservice', 'productcatalogservice'),            ('checkoutservice', 'paymentservice'), ('checkoutservice', 'productcatalogservice')        ]        print('Env: 228D obs, 10D action')        print(f'  Controlled (10): {self.controlled}')        print(f'  Edges (10): {self.obs_edges}')    def _normalize(self, values, mins, maxs):        ranges = np.asarray(maxs, dtype=np.float32) - np.asarray(mins, dtype=np.float32)        ranges[ranges == 0] = 1.0        return np.clip((np.asarray(values, dtype=np.float32) - np.asarray(mins, dtype=np.float32)) / ranges, 0.0, 1.0)    def _forecast(self):        if self.predictor is None:            return np.zeros((6, self.n_all, 2), dtype=np.float32)        try:            window_rates = self.ep['rates'][self.t - 18: self.t]            x_list, e_list = [], []            for i in range(18):                tick_rate = window_rates[i]                node_feats = [self._get_service_features(j, tick_rate) for j in range(self.n_all)]                x_list.append(np.stack(node_feats))                e_list.append(np.stack([[0.1, 0.5, 0.01, 50.0] for _ in self.obs_edges]))            x_raw = np.stack(x_list)[None, ...]            e_raw = np.stack(e_list)[None, ...]            forecast = self.predictor.predict(x_raw, e_raw, batch_size=1)[0]            return np.maximum(forecast, 0.0)        except Exception as e:            print(f'Forecast failed: {e}, zeros')            return np.zeros((6, self.n_all, 2), dtype=np.float32)    def _get_service_features(self, svc_idx, rate):        if self.state is None:            return np.zeros(13, dtype=np.float32)        svc = self.all_services[svc_idx]        return np.array([            self.state.get('cpu_cores', {}).get(svc, 0.5),            self.state.get('mem_mib', {}).get(svc, 1024.0),            self.state.get('cpu_util_request', {}).get(svc, 0.5),            self.state.get('cpu_throttle_ratio', {}).get(svc, 0.0),            self.state.get('cpu_sidecar_cores', {}).get(svc, 0.1),            self.state.get('latency_p50_ms', {}).get(svc, 50.0),            self.state.get('latency_p95_ms', {}).get(svc, 100.0),            self.state.get('error_rate', {}).get(svc, 0.0),            self.state.get('rps_in', {}).get(svc, rate / self.n_all),            self.state.get('rps_in_delta', {}).get(svc, 0.0),            self.state.get('rps_per_replica', {}).get(svc, rate / self.n_all / max(1, self.state.get('replicas', {}).get(svc, 1))),            float(self.state.get('replicas', {}).get(svc, 1)),            float(self.state.get('restarts_delta', {}).get(svc, 0)),        ], dtype=np.float32)    def _obs(self, state):        curr = []        for svc in self.all_services:            for field in self.current_fields:                curr.append(state.get(field, {}).get(svc, 0.0))        curr_arr = np.array(curr, dtype=np.float32).reshape(self.n_all, 13)        curr_n = self._normalize(curr_arr, self.feature_min, self.feature_max).ravel()        forecast = self._forecast()        fut = []        for h in (2, 5):            sv = forecast[h, :, :2] if h < forecast.shape[0] else np.zeros((self.n_all, 2), dtype=np.float32)            fut.append(self._normalize(sv.ravel(), np.array([0.0,0.0]), np.array([100.0,8.0])).ravel())        future_n = np.concatenate(fut)        graph = []        for edge in self.obs_edges:            graph.extend(state.get('edge_features', {}).get(edge, [0.1, 0.5, 0.01, 50.0]))        graph_n = self._normalize(np.array(graph, dtype=np.float32), np.zeros(40), np.full(40, 100.0))        rf = np.clip(np.array([(self.reward_p95_ema if self.reward_p95_ema is not None else CFG.lat_slo_ms) / CFG.lat_slo_ms], dtype=np.float32), 0.0, 1.0)        obs = np.concatenate([curr_n, future_n, graph_n, rf]).astype(np.float32)        if len(obs) < 228: obs = np.pad(obs, (0, 228 - len(obs)), 'constant')        elif len(obs) > 228: obs = obs[:228]        return obs    def reset(self, seed=None, options=None):        super().reset(seed=seed)        if seed is not None: self.rng = np.random.default_rng(seed)        self.twin = _MockTwin(self.rng.integers(10000), self.all_services)        if self.sequential:            self.ep = self.episodes[self.episode_cursor % len(self.episodes)]            self.episode_cursor += 1        else:            self.ep = self.episodes[int(self.rng.integers(len(self.episodes)))]        if len(self.ep['rates']) <= 18:            raise ValueError('Episode shorter than history window')        self.twin.reset()        self.t = 0        self.reward_p95_ema = 300.0        self.episode_step = 0        state = None        for _ in range(18):            state = self.twin.tick(self.ep['rates'][self.t])            self.t += 1        self.state = state        return self._obs(state), {'episode': self.ep.get('key', 'unknown')}    def step(self, action):        action = np.asarray(action, dtype=int)        if hasattr(self.twin, 'apply_action_11'):            action_result = self.twin.apply_action_11(action)        else:            action_result = self.twin.apply_action(action)        state = self.state        reward = 0.0        terminated = False        truncated = False        for _ in range(3):            if self.t >= len(self.ep['rates']): break            state = self.twin.tick(self.ep['rates'][self.t])            self.t += 1            self.episode_step += 1            f_raw = state.get('p95_raw', {}).get('frontend', 300.0)            cnt = int(state.get('span_count', {}).get('frontend', 0))            if cnt > 0:                a = 0.1                self.reward_p95_ema = (1 - a) * self.reward_p95_ema + a * f_raw            sla_p = max(0.0, (self.reward_p95_ema - 300.0) / 300.0)            tot_rep = float(np.sum(state.get('ready', {}).values()))            cost_r = tot_rep / max(1.0, tot_rep + 10.0)            rps_c = np.mean(self.ep['rates'][max(0, self.t - 5): self.t])            d_rps = float(np.mean(self.ep['rates']))            tput_r = float(rps_c / max(d_rps, 1e-6))            lf = state.get('features', {})            is_ood = False            if self.lof_detector is not None and lf is not None:                try:                    lv = np.array([lf.get(s, 0.0) for s in self.controlled], dtype=np.float32).reshape(1, -1)                    if self.lof_detector.predict(lv)[0] == -1:                        is_ood = True                except Exception:                    pass            if is_ood:                reward = -100.0                info = {'frontend_p95_ms': float(self.reward_p95_ema), 'total_replicas': int(tot_rep),                        'ood_flag': True, 'sla_penalty': float(sla_p), 'cost_ratio': float(cost_r),                        'throughput_ratio': float(tput_r)}                self.state = state                return self._obs(state), float(reward), terminated, True, info            reward += 0.0 * tput_r - 5.0 * sla_p - 1.0 * cost_r        self.state = state        terminated = self.t >= len(self.ep['rates'])        info = {'frontend_p95_ms': float(self.reward_p95_ema),                'total_replicas': int(np.sum(state.get('ready', {}).values())),                'ood_flag': False, 'sla_penalty': float(sla_p),                'cost_ratio': float(cost_r), 'throughput_ratio': float(tput_r)}        return self._obs(state), float(reward), terminated, truncated, info

## 6. Sanity check và PPO multi-seed


In [ ]:
class _MockTwin:
    """Placeholder twin generating synthetic state from episode rates."""
    def __init__(self, seed, all_services):
        self.rng = np.random.default_rng(seed)
        self.all_services = all_services
        self.n = len(all_services)
        self.reset()

    def reset(self):
        self.desired = {s: 1 for s in self.all_services}
        self.ready = {s: 1 for s in self.all_services}
        self.cooldown = {s: 0 for s in self.all_services}
        self.pending = {s: [] for s in self.all_services}
        self.last_delta = {s: 0 for s in self.all_services}

    def apply_action(self, action_10):
        controlled = CFG.controlled
        changes = 0
        for i, svc in enumerate(controlled):
            delta = int(action_10[i]) - 1
            if delta == 0:
                continue
            target = int(np.clip(self.desired[svc] + delta, CFG.min_replicas, CFG.max_replicas))
            if target != self.desired[svc]:
                self.desired[svc] = target
                self.cooldown[svc] = CFG.cooldown_ticks
                changes += 1
        return {'changes': changes, 'reversals': 0, 'blocked': 0}

    def apply_action_11(self, action_10):
        return self.apply_action(action_10)

    def tick(self, offered_rps):
        for s in self.all_services:
            self.cooldown[s] = max(0, self.cooldown[s] - 1)
            self.pending[s] = [d - CFG.tick_sec for d in self.pending[s]]
            completed = sum(1 for d in self.pending[s] if d <= 0)
            self.ready[s] += completed
            self.pending[s] = [d for d in self.pending[s] if d > 0]
        per_svc = offered_rps / self.n
        state = {}
        for s in self.all_services:
            rep = max(1, self.ready[s])
            util = min(1.0, per_svc / (rep * 25.0))
            cpu = 0.5 + 0.5 * util + 0.1 * self.rng.random()
            mem = 512.0 + 512.0 * util
            lat_p50 = 20.0 + 80.0 * util ** 2
            lat_p95 = lat_p50 * 2.5 + 20.0 * self.rng.random()
            err = 0.001 * max(0, util - 0.8) ** 2
            state[s] = {
                'cpu_cores': cpu * rep,
                'mem_mib': mem * rep,
                'cpu_util_request': cpu / rep,
                'cpu_throttle_ratio': max(0, util - 0.9) * 0.1,
                'cpu_sidecar_cores': 0.1 * rep,
                'latency_p50_ms': lat_p50,
                'latency_p95_ms': lat_p95,
                'error_rate': err,
                'rps_in': per_svc,
                'rps_in_delta': 0.0,
                'rps_per_replica': per_svc / rep,
                'replicas': float(rep),
                'restarts_delta': float(self.last_delta.get(s, 0)),
            }
        state['features'] = {s: state[s]['cpu_util_request'] for s in self.all_services}
        state['p95_raw'] = {s: state[s]['latency_p95_ms'] for s in self.all_services}
        state['span_count'] = {s: max(1, int(per_svc * 0.1)) for s in self.all_services}
        state['p95'] = {s: state[s]['latency_p95_ms'] for s in self.all_services}
        state['error'] = {s: state[s]['error_rate'] for s in self.all_services}
        state['edge_features'] = {e: [0.1, 0.5, 0.01, 50.0] for e in [
            ('frontend', 'productcatalogservice'), ('frontend', 'currencyservice'),
            ('frontend', 'cartservice'), ('frontend', 'recommendationservice'),
            ('frontend', 'shippingservice'), ('frontend', 'checkoutservice'),
            ('frontend', 'adservice'), ('recommendationservice', 'productcatalogservice'),
            ('checkoutservice', 'paymentservice'), ('checkoutservice', 'productcatalogservice')]}
        return state

def build_episodes_from_dataset(dataset_root, patterns=('mmpp', 'nhpp', 'onoff', 'pareto'), runs_per_pattern=8):
    episodes = []
    for pattern in patterns:
        for run in range(1, runs_per_pattern + 1):
            run_dir = Path(dataset_root) / pattern / f'run_{run:02d}'
            edge_file = run_dir / 'edge_metrics.csv'
            if not edge_file.exists():
                continue
            try:
                df = pd.read_csv(edge_file)
                if 'timestamp' not in df.columns or 'call_count' not in df.columns:
                    continue
                rates = df.groupby('timestamp')['call_count'].sum().values.astype(float)
                if len(rates) < 23:
                    continue
                episodes.append({
                    'rates': rates,
                    'pattern': pattern,
                    'run': run,
                    'key': f'{pattern}/run_{run:02d}',
                })
            except Exception as e:
                print(f'Skipping {pattern}/run_{run:02d}: {e}')
    print(f'Loaded {len(episodes)} episodes from dataset')
    return episodes


KAGGLE_INPUT = '/kaggle/input/thesis-gnn-rl-autoscaling'
DATASET_ROOT = f'{KAGGLE_INPUT}/dataset_raw'
TRAIN_EPISODES = build_episodes_from_dataset(DATASET_ROOT)
if not TRAIN_EPISODES:
    print('No dataset episodes — generating synthetic')
    TRAIN_EPISODES = [
        {'rates': np.random.poisson(50, 200).astype(float), 'pattern': 'synthetic', 'run': 0, 'key': 'syn/0'},
        {'rates': np.random.poisson(30, 200).astype(float), 'pattern': 'synthetic', 'run': 1, 'key': 'syn/1'},
    ]
n_train = int(0.8 * len(TRAIN_EPISODES))
VAL_EPISODES = TRAIN_EPISODES[n_train:]
TRAIN_EPISODES = TRAIN_EPISODES[:n_train]
print(f'Train: {len(TRAIN_EPISODES)}, Val: {len(VAL_EPISODES)}')

from stable_baselines3.common.env_checker import check_env
env = MicroserviceDigitalTwin(TRAIN_EPISODES, SEED)
check_env(env, warn=True)
obs, info = env.reset()
done = False
steps = 0
while not done and steps < 20:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    assert np.isfinite(obs).all() and np.isfinite(reward)
    done = terminated or truncated
    steps += 1
print(f'Sanity check passed: {steps} steps, obs shape {obs.shape}, reward {reward:.3f}')

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

GATGRU_CHECKPOINT = f'{KAGGLE_INPUT}/checkpoints/gatgru_forecastor.pt'
LOF_MODEL_PATH = f'{KAGGLE_INPUT}/checkpoints/lof_model.pkl'

PREDICTOR = None
try:
    sys.path.insert(0, f'{KAGGLE_INPUT}/modeling-common')
    from gatgru_inference import GATGRUPredictor, EXPECTED_EDGES
    PREDICTOR = GATGRUPredictor(GATGRU_CHECKPOINT, EXPECTED_EDGES, device=str(DEVICE))
    print(f'GAT-GRU loaded: horizon={PREDICTOR.horizon}')
except Exception as e:
    print(f'GAT-GRU not available: {e}')

LOF_DETECTOR = None
try:
    LOF_DETECTOR = joblib.load(LOF_MODEL_PATH)
    print(f'LOF detector loaded')
except Exception as e:
    print(f'LOF model not available: {e}')


def make_env(episodes, seed, sequential=False):
    def _init():
        return MicroserviceDigitalTwin(
            episodes=episodes, seed=seed, sequential=sequential,
            predictor=PREDICTOR, lof_detector=LOF_DETECTOR)
    return _init


train_env = DummyVecEnv([make_env(TRAIN_EPISODES, SEED, sequential=False)])
val_env = DummyVecEnv([make_env(VAL_EPISODES, SEED + 10000, sequential=True)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
val_env = VecNormalize(val_env, training=False, norm_obs=True, norm_reward=False, clip_obs=10.0)

model = PPO(
    'MlpPolicy', train_env,
    learning_rate=2e-4, n_steps=2048, batch_size=256,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.05,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=1, device=str(DEVICE), seed=SEED)

import os as _os
_os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
_os.makedirs('/kaggle/working/logs', exist_ok=True)

eval_callback = EvalCallback(
    val_env, best_model_save_path='/kaggle/working/checkpoints/',
    log_path='/kaggle/working/logs/', eval_freq=10000,
    n_eval_episodes=4, deterministic=True, render=False)
checkpoint_callback = CheckpointCallback(
    save_freq=10000, save_path='/kaggle/working/checkpoints/', name_prefix='ppo_autoscaler')

print('Starting PPO training...')
model.learn(
    total_timesteps=300000,
    callback=[eval_callback, checkpoint_callback],
    progress_bar=True)

model.save('/kaggle/working/ppo_autoscaler_final')
train_env.save('/kaggle/working/checkpoints/vecnormalize_train.pkl')
print('Training complete. Model saved to /kaggle/working/')

In [ ]:
# Final evaluation on validation set
def evaluate_agent(model, env, n_episodes=4):
    rewards, p95s, replicas_list, ood_counts = [], [], [], []
    for ep_idx in range(n_episodes):
        obs = env.reset()
        ep_r, ep_p95, ep_rep = 0.0, [], []
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_r += float(reward)
            if 'frontend_p95_ms' in info:
                ep_p95.append(info['frontend_p95_ms'])
            if 'total_replicas' in info:
                ep_rep.append(info['total_replicas'])
            if info.get('ood_flag', False):
                ood_counts.append(1)
            done = terminated[0] if isinstance(terminated, __import__('numpy').ndarray) else terminated
        rewards.append(ep_r)
        p95s.append(float(np.mean(ep_p95)) if ep_p95 else 0.0)
        replicas_list.append(float(np.mean(ep_rep)) if ep_rep else 0.0)
    return {
        'mean_reward': float(np.mean(rewards)),
        'mean_p95_ms': float(np.mean(p95s)),
        'mean_replicas': float(np.mean(replicas_list)),
        'ood_events': int(len(ood_counts)),
    }


print('Running final evaluation...')
eval_results = evaluate_agent(model, val_env, n_episodes=4)
for k, v in eval_results.items():
    print(f'  {k}: {v}')

## Điều kiện báo cáo
Không dùng kết quả nếu Digital Twin chưa đạt gate theo từng service/scenario, policy đi ngoài replica 1–3, hoặc best checkpoint không được ghép đúng `VecNormalize`. Test calibration chỉ được mở sau khi model/twin/reward và checkpoint-selection protocol đã đóng băng.
